## 🔧 Agents with Tools

**Goal**: Enhance agents with built-in tools — **Code Interpreter**, **File Search**, **Web Search**, and custom **Function Tools** — registered server-side in Foundry with full **tracing**, then used via `FoundryAgent`.

- `CodeInterpreterTool` — data analysis & charts (server-side)
- `FileSearchTool` — RAG over uploaded documents (server-side)
- `WebSearchTool` — real-time web search (server-side)
- `FunctionTool` + `@tool` decorator — custom Python functions
- **Tracing** — all interactions visible in Foundry portal

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential
from agent_framework import tool

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    FileSearchTool,
    WebSearchTool,
    FunctionTool as AzureFunctionTool,
)
from agent_framework.foundry import FoundryAgent

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

print(f"📡 Project endpoint: {PROJECT_ENDPOINT}")
print(f"🧠 Model: {MODEL}")
print("✅ Connected to Foundry")

📡 Project endpoint: https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model: gpt-4o
✅ Connected to Foundry


---
### Part A — Code Interpreter 🧮
Register an agent with `CodeInterpreterTool` in Foundry, then use it via `FoundryAgent`.
The agent writes and executes Python code to analyze data and create charts — all traced server-side.

In [2]:
# ── A1. Register Code Interpreter agent in Foundry ───────────────────────────
CODE_AGENT_NAME = "DataAnalyst-af"

code_agent_version = project.agents.create_version(
    agent_name=CODE_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a data analyst. When given data, write and execute Python "
            "code to analyze it. Always produce a chart when asked for visualization. "
            "Explain your findings clearly."
        ),
        tools=[CodeInterpreterTool()],
    ),
)

# Connect via Agent Framework
code_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=code_agent_version.name,
    agent_version=str(code_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {code_agent_version.name} v{code_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(code_agent).__name__}")

✅ Agent registered in Foundry: DataAnalyst-af v1
🤖 FoundryAgent connected: FoundryAgent


In [3]:
# ── A2. Ask for analysis with code execution ─────────────────────────────────
from agent_framework import Content

result = await code_agent.run(
    "Calculate the first 20 Fibonacci numbers and create a line chart showing their growth."
)
print(f"🗨️  Agent:\n{result}")

# Inspect generated code
for message in result.messages:
    code_blocks = [c for c in message.contents if c.type == "code_interpreter_tool_call"]
    outputs = [c for c in message.contents if c.type == "code_interpreter_tool_result"]
    if code_blocks:
        for content in (code_blocks[0].inputs or []):
            if isinstance(content, Content) and content.type == "text":
                print(f"\n📝 Generated code:\n{content.text}")
                break
    if outputs:
        for out in (outputs[0].outputs or []):
            if isinstance(out, Content) and out.type == "text":
                print(f"\n📊 Execution output:\n{out.text}")

print(f"\n🔍 Check Foundry portal → Tracing to see this interaction!")

🗨️  Agent:
The chart displays the growth of the first 20 Fibonacci numbers. The Fibonacci sequence begins with 0 and 1, and each subsequent number is the sum of the previous two.

Key observations:

- The growth is exponential, evident from the steep curve as the index increases. 
- Early numbers grow slowly, but as you proceed further along the sequence, the numbers rapidly increase.
- This pattern reflects the nature of the Fibonacci sequence, where each term builds on the cumulative total of its predecessors, leading to rapid growth.

This visualization helps to understand how quickly Fibonacci numbers can grow beyond their initial, relatively small values.

📝 Generated code:
import matplotlib.pyplot as plt

# Function to calculate the first n Fibonacci numbers
def fibonacci_sequence(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence

# Calculate the first 20 Fibonacci numbers
first_

---
### Part B — File Search (RAG) 📂
Uploaded product docs into a Vector Store, register an agent with `FileSearchTool` in Foundry, then query via `FoundryAgent`.

In [4]:
# ── B2. Register File Search agent in Foundry ────────────────────────────────
FILE_AGENT_NAME = "ProductSpecialist-af"
vectorstoredid = "vs_ZK0XW6cotxNUoZ8rBAqWShcW"

file_agent_version = project.agents.create_version(
    agent_name=FILE_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Contoso product specialist. Use the file search tool to "
            "look up product details, pricing, and policies from the company "
            "knowledge base. Always cite the source."
        ),
        tools=[FileSearchTool(vector_store_ids=[vectorstoredid])],
    ),
)

# Connect via Agent Framework
file_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=file_agent_version.name,
    agent_version=str(file_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {file_agent_version.name} v{file_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(file_agent).__name__}")

✅ Agent registered in Foundry: ProductSpecialist-af v1
🤖 FoundryAgent connected: FoundryAgent


In [5]:
# ── B3. Ask questions grounded in the uploaded doc ───────────────────────────
for question in [
    "What is the price of the Contoso Widget Pro?",
    "Compare Widget Pro vs Widget Lite.",
    "What is the return policy for hardware?",
]:
    resp = await file_agent.run(question)
    print(f"\n❓ {question}")
    print(f"🗨️  {resp}")

print(f"\n🔍 Check Foundry portal → Tracing to see these interactions!")


❓ What is the price of the Contoso Widget Pro?
🗨️  The price of the Contoso Widget Pro is $49.99.

❓ Compare Widget Pro vs Widget Lite.
🗨️  Here's a comparison between Contoso Widget Pro and Widget Lite:

**Contoso Widget Pro**:
- **Price**: $49.99
- **Description**: Premium widget with titanium-reinforced housing and smart-connect technology. Ideal for enterprise deployments.
- **Warranty**: 3 years
- **Weight**: 250g

**Contoso Widget Lite**:
- **Price**: $19.99
- **Description**: Lightweight, budget-friendly widget for personal and small-team use. Easy to set up in under 5 minutes.
- **Warranty**: 1 year
- **Weight**: 120g

The Widget Pro is designed for enterprises with enhanced durability and connectivity features, while the Widget Lite is intended for simpler, personal or small team uses with a focus on ease of setup and affordability.

❓ What is the return policy for hardware?
🗨️  The return policy for hardware at Contoso allows returns within 30 days of purchase, provided the 

---
### Part C — Web Search 🌐
Register an agent with `WebSearchTool` in Foundry to fetch real-time info from the web, all traced server-side.

In [6]:
# ── C1. Register Web Search agent in Foundry ─────────────────────────────────
WEB_AGENT_NAME = "ResearchAssistant-af"

web_agent_version = project.agents.create_version(
    agent_name=WEB_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a research assistant. Use web search to find the latest "
            "information and always include source URLs in your response."
        ),
        tools=[WebSearchTool()],
    ),
)

# Connect via Agent Framework
web_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=web_agent_version.name,
    agent_version=str(web_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {web_agent_version.name} v{web_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(web_agent).__name__}")

result = await web_agent.run(
    "What are the top AI announcements from Microsoft Build this year?"
)
print(f"\n🗨️  Agent: {result}")
print(f"\n🔍 Check Foundry portal → Tracing to see this interaction!")

✅ Agent registered in Foundry: ResearchAssistant-af v1
🤖 FoundryAgent connected: FoundryAgent

🗨️  Agent: Here’s what I found regarding the **top AI-focused announcements at Microsoft Build 2026**—to be held on **June 2–3, 2026** in San Francisco, largely focused on AI and developer tools. Note that as of today, **April 2, 2026**, Build hasn't occurred yet, so public details are limited to event previews and expectations. Below is the latest official information; once the event takes place, I can provide a full breakdown of announcements in detail.

---

##  Microsoft Build 2026: Focus & Format

- **Dates & Venue**  
  Microsoft Build 2026 is confirmed for **June 2–3, 2026**, at the **Fort Mason Center** in San Francisco, marking the first time the conference is held in the city since 2016 ([financialexpress.com](https://www.financialexpress.com/life/technology-microsoft-build-2026-dates-announced-everything-to-know-about-this-ai-focused-developer-event-at-san-francisco-4163205/)).

- 

---
### Part D — Custom Function Tools 🛠️
Register an agent with function tool schemas in Foundry, then pass the Python implementations to `FoundryAgent`.
The schema lives server-side (traced), while the function logic runs locally.

In [7]:
# ── D1. Define function tools ────────────────────────────────────────────────
from typing import Annotated
from pydantic import Field


@tool(description="Look up current inventory count for a product SKU.")
def check_inventory(
    sku: Annotated[str, Field(description="The product SKU, e.g. CW-PRO-001")],
) -> str:
    """Simulated inventory lookup."""
    inventory = {
        "CW-PRO-001": 142,
        "CW-LITE-002": 580,
        "CC-CLOUD-003": 999,
    }
    count = inventory.get(sku.upper(), None)
    if count is None:
        return f"SKU '{sku}' not found in inventory."
    return f"SKU {sku}: {count} units in stock."


@tool(description="Submit a support ticket with a description.")
def create_support_ticket(
    title: Annotated[str, Field(description="Short summary of the issue")],
    description: Annotated[str, Field(description="Detailed description")],
) -> str:
    """Simulated ticket creation."""
    import random
    ticket_id = f"TKT-{random.randint(1000, 9999)}"
    return f"✅ Ticket {ticket_id} created: '{title}'"


print("🛠️  Custom tools defined: check_inventory, create_support_ticket")

🛠️  Custom tools defined: check_inventory, create_support_ticket


In [8]:
from collections.abc import Sequence as SeqABC, Mapping as MapABC
from typing import Any
from agent_framework import Message, FunctionTool as AFFunctionTool
from agent_framework.observability import ChatTelemetryLayer
from agent_framework._compaction import CompactionStrategy, TokenizerProtocol
from agent_framework.foundry import RawFoundryAgentChatClient
# Need these imports for the MRO
from agent_framework import FunctionInvocationLayer, ChatMiddlewareLayer

# ── D2. Register Function Tool agent in Foundry ──────────────────────────────
TOOL_AGENT_NAME = "ContosoHelper-af"

# Workaround: Foundry Responses API rejects 'tools' in the request body when
# an agent_reference is set. Subclass to strip tools from the request while
# keeping them for local function invocation.
class _FixedFoundryClient(
    FunctionInvocationLayer,
    ChatMiddlewareLayer,
    ChatTelemetryLayer,
    RawFoundryAgentChatClient,
):
    async def _prepare_options(self, messages: SeqABC[Message], options: MapABC[str, Any], **kwargs: Any):
        run_options = await super()._prepare_options(messages, options, **kwargs)
        run_options.pop("tools", None)  # tools live on the agent definition, not the request
        return run_options

# Need these imports for the MRO
from agent_framework import FunctionInvocationLayer, ChatMiddlewareLayer

# Define function schemas for the Foundry agent definition
tool_agent_version = project.agents.create_version(
    agent_name=TOOL_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Contoso support agent. Use the inventory tool to check stock "
            "levels and the ticket tool to create support tickets when needed."
        ),
        tools=[
            AzureFunctionTool(
                name="check_inventory",
                description="Look up current inventory count for a product SKU.",
                parameters={
                    "type": "object",
                    "properties": {
                        "sku": {"type": "string", "description": "The product SKU, e.g. CW-PRO-001"},
                    },
                    "required": ["sku"],
                },
                strict=False,
            ),
            AzureFunctionTool(
                name="create_support_ticket",
                description="Submit a support ticket with a description.",
                parameters={
                    "type": "object",
                    "properties": {
                        "title": {"type": "string", "description": "Short summary of the issue"},
                        "description": {"type": "string", "description": "Detailed description"},
                    },
                    "required": ["title", "description"],
                },
                strict=False,
            ),
        ],
    ),
)

# Connect via Agent Framework — pass local function implementations
tool_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=tool_agent_version.name,
    agent_version=str(tool_agent_version.version),
    credential=get_credential(),
    tools=[check_inventory, create_support_ticket],
    client_type=_FixedFoundryClient,
)

print(f"✅ Agent registered in Foundry: {tool_agent_version.name} v{tool_agent_version.version}")
print(f"🤖 FoundryAgent connected with local tool implementations")

# The agent will autonomously call the right tool
result = await tool_agent.run("How many Widget Pro units (CW-PRO-001) do we have in stock?")
print(f"\n🗨️  {result}")

result2 = await tool_agent.run(
    "Create a support ticket: 'Cloud Connector sync failing in EU region' "
    "with description 'OTA updates timing out after 30 seconds for EU customers.'"
)
print(f"\n🗨️  {result2}")
print(f"\n🔍 Check Foundry portal → Tracing to see these interactions!")

✅ Agent registered in Foundry: ContosoHelper-af v1
🤖 FoundryAgent connected with local tool implementations

🗨️  We currently have 142 Widget Pro units (CW-PRO-001) in stock.

🗨️  The support ticket has been created with the ID TKT-7269 for the issue: 'Cloud Connector sync failing in EU region.'

🔍 Check Foundry portal → Tracing to see these interactions!


In [9]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
# Delete agents from Foundry
for name in [CODE_AGENT_NAME, FILE_AGENT_NAME, WEB_AGENT_NAME, TOOL_AGENT_NAME]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🧹 Deleted agent: {name}")
    except Exception as e:
        print(f"⚠️  Could not delete {name}: {e}")

# Delete vector store and file
try:
    oai.vector_stores.delete(vector_store.id)
    oai.files.delete(file.id)
    print("🧹 Vector store and file deleted")
except Exception:
    pass

print("✅ Cleanup complete")

🧹 Deleted agent: DataAnalyst-af
🧹 Deleted agent: ProductSpecialist-af
🧹 Deleted agent: ResearchAssistant-af
🧹 Deleted agent: ContosoHelper-af
✅ Cleanup complete


---
### ✅ What you demonstrated
| Capability | Foundry Definition | Agent Framework Feature |
|---|---|---|
| Data analysis & charts | `CodeInterpreterTool()` on `PromptAgentDefinition` | Runs server-side via `FoundryAgent` |
| Document Q&A (RAG) | `FileSearchTool(vector_store_ids=[...])` | Runs server-side via `FoundryAgent` |
| Real-time web knowledge | `WebSearchTool()` | Runs server-side via `FoundryAgent` |
| Custom function tools | `FunctionTool` schema on definition | `@tool` impl passed to `FoundryAgent(tools=[...])` |
| Auto tool selection | Agent autonomously picks the right tool | Same behavior, fully traced |
| Server-side tracing | All interactions visible in Foundry portal → Tracing | |

**Next →** [03_tool_mcp_integration.ipynb](./03_tool_mcp_integration.ipynb) — MCP tools and agent-as-tool composition.